In [ ]:
storage_account = 'adlsretailpulse<initials>'
container       = 'retaildata'

df_raw = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .csv(f'abfss://{container}@{storage_account}.dfs.core.windows.net/raw/customers/')

print(f'Row count: {df_raw.count():,}')   # Expected: 50,000
df_raw.printSchema()
df_raw.show(5)

In [4]:
from pyspark.sql.functions import col, sum as _sum

# Check nulls across all columns
null_counts = df_raw.select([
    _sum(col(c).isNull().cast('int')).alias(c)
    for c in df_raw.columns
])
null_counts.show()

# Check distinct values for key columns
print('Cities:')
df_raw.groupBy('city').count().show()

print('Loyalty Tiers:')
df_raw.groupBy('loyalty_tier').count().show()

print('Gender:')
df_raw.groupBy('gender').count().show()

# Age range
from pyspark.sql.functions import min as _min, max as _max
df_raw.select(_min('age'), _max('age')).show()
# Expected: min=18, max=69

In [5]:
from pyspark.sql.functions import (
    to_date, when, col, trim, upper, initcap
)

df_clean = (df_raw
    # Fix signup_date — format in file is d/MM/yyyy (e.g. 1/01/2018)
    .withColumn('signup_date',
        to_date(col('signup_date'), 'd/MM/yyyy'))

    # Standardise text columns — remove accidental spaces
    .withColumn('first_name',   trim(col('first_name')))
    .withColumn('city',         trim(initcap(col('city'))))
    .withColumn('gender',       trim(upper(col('gender'))))
    .withColumn('loyalty_tier', trim(initcap(col('loyalty_tier'))))

    # Ensure correct types
    .withColumn('customer_id', col('customer_id').cast('int'))
    .withColumn('age',         col('age').cast('int'))

    # Filter out invalid ages just in case
    .filter(col('age').between(18, 100))

    # Drop duplicates on customer_id
    .dropDuplicates(['customer_id'])
)

print(f'Clean row count: {df_clean.count():,}')
# Expected: 50,000 (no rows lost — data is clean)
df_clean.printSchema()
df_clean.show(5)

In [6]:
from pyspark.sql.functions import (
    when, col, year, months_between, current_date, round as _round
)

df_enriched = (df_clean
    # Age group segmentation — useful for customer analytics
    .withColumn('age_group',
        when(col('age').between(18, 30), '18-30')
       .when(col('age').between(31, 50), '31-50')
       .when(col('age').between(51, 100), '51+')
       .otherwise('Unknown'))

    # Loyalty score — numeric version of loyalty tier for ML later
    .withColumn('loyalty_score',
        when(col('loyalty_tier') == 'Platinum', 4)
       .when(col('loyalty_tier') == 'Gold',     3)
       .when(col('loyalty_tier') == 'Silver',   2)
       .when(col('loyalty_tier') == 'Bronze',   1)
       .otherwise(0))

    # Customer tenure in years since signup
    .withColumn('tenure_years',
        _round(months_between(current_date(), col('signup_date')) / 12, 1))

    # Signup year — useful for cohort analysis
    .withColumn('signup_year', year(col('signup_date')))
)

print('Age group distribution:')
df_enriched.groupBy('age_group').count().orderBy('age_group').show()

print('Loyalty distribution:')
df_enriched.groupBy('loyalty_tier', 'loyalty_score').count().orderBy('loyalty_score').show()

In [8]:
silver_path = (
    f'abfss://{container}@{storage_account}'
    f'.dfs.core.windows.net/curated/customers/'
)

df_enriched.write \
    .mode('overwrite') \
    .parquet(silver_path)

print('✅ Customers Silver layer written successfully!')

# Verify
df_verify = spark.read.parquet(silver_path)
print(f'Verified row count: {df_verify.count():,}')
# Expected: 50,000
df_verify.printSchema()